# Technique: data poisoning with image data

Based on the following paper:

**Goodfellow, Ian J., Jonathon Shlens, and Christian Szegedy. "Explaining and harnessing adversarial examples." arXiv preprint arXiv:1412.6572 (2014).**

In this project we'll examine some popular attacks which apply data-poisoning methods to maximize the loss of a target neural network.

This **white-box** technique (FGSM) takes in test data and modifies the individual pixel values according to parameter $\epsilon \in [0,1]$ and a normalization scheme. We'll compare the accuracy between the in-class CNN and the victim CNN.

In [29]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
import numpy as np
import torch.nn.functional as F

'''
CNN.ipynb
----------

Implements a CNN (using the implementation from class with some tweaks.) Performs
rudimentary data analysis using the MNIST dataset, and then attempts to perturb
the dataset so our CNN will misclassify the image. The perturbations should be as minor
and undetectable as possible.

We want to search for a perturbation which does not heinously alter our image.
Rotations and translations, pixel displacements, etc.

Fast Gradient Sign Attack which maximizes loss with respect to input data

'''
def fgsm(loss_gradient, img_data, perturbation_parameter):
  '''
  Takes in perturbation parameter epsilon and returns an
  altered image coordinate. This is a linear-style attack which
  can be applied to nonlinear data.
  '''
  # get sign of loss gradient
  sign_loss_gradient = loss_gradient.sign()

  # pixel-wise adjustment
  new_img_data = img_data + perturbation_parameter * sign_loss_gradient

  # clipping
  new_img_data = torch.clamp(new_img_data, 0, 1)

  return new_img_data

def denorm(batch, mean=[0.1307], std=[0.3081]):
  if isinstance(mean, list):
    mean = torch.tensor(mean).to(device)
  if isinstance(std, list):
    std = torch.tensor(std).to(device)

  return batch * std.view(1, -1, 1, 1) + mean.view(1,-1, 1, 1)

def perturbator(model, device, test_loader, epsilon):
  '''
  This is a bad-faith test function which injects modified test data into the dataset.
  '''
  ex = []
  # Calculate Accuracy
  correct = 0
  # Iterate through test dataset
  for images, labels in test_loader:

    # Load images
    images,labels = images.to(device), labels.to(device)
    images.requires_grad = True

    # Forward pass only to get logits/output
    outputs = model(images)

    # Get predictions from the max log-probability
    init_predicted = outputs.max(1, keepdim=True)[1]

    if predicted.item() != labels.item():
      continue

    loss = F.nll_loss(outputs, labels)

    model.zero_grad()

    loss.backward()

    data_grad = images.grad.data

    data_denorm = denorm(images)

    # mount the attack
    pdata = fgsm(data_grad, data_denorm, epsilon)

    # normalize perturbed data
    pdata_normalized = transforms.Normalize((0.1307), (0.3081))(pdata)

    outputs = model(pdata_normalized)

    final_predicted = outputs.max(1, keepdim=True)[1]
    if final_predicted.item() == labels.item():
      correct += 1
      if epsilon == 0 and len(ex) < 5:
        adv_ex = pdata.squeeze().detach().cpu().numpy()
        ex.append((init_predicted.item(), final_predicted.item(), adv_ex))
    else:
      if len(ex) < 5:
        adv_ex = pdata.squeeze().detach().cpu().numpy()
        ex.append((init_predicted.item(), final_predicted.item(), ex))

  final_accuracy = correct/float(len(test_loader))

  # Print loss
  print('Epsilon_val: {}. Loss: {}. Accuracy: {}'.format(epsilon, loss.item(), accuracy))
  return final_accuracy, adv_ex

train_dataset = dsets.MNIST(root='./data',
                            train=True,
                            transform=transforms.ToTensor(),
                            download=True)

test_dataset = dsets.MNIST(root='./data',
                           train=False,
                           transform=transforms.ToTensor())

# make dataset iterable
batch_size = 1
n_iters = 3000
num_epochs = 2
num_epochs = int(num_epochs)
#print(num_epochs)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=100,
                                           shuffle=True)

test_loader = torch.utils.data.DataLoader(dataset=test_dataset,
                                          batch_size=batch_size,
                                          shuffle=True)
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
# this will be the NN that we'll be attacking
class victim_CNN(nn.Module):
  def __init__(self):
    super(victim_CNN, self).__init__()

    # Convolution 1
    self.cnn1 = nn.Conv2d(in_channels = 1, out_channels = 16, kernel_size=5, stride=1, padding=2)
    self.relu1 = nn.ReLU()

    # Max pool 1
    self.maxpool1 = nn.MaxPool2d(kernel_size = 2)

    # Convolution 2
    self.cnn2 = nn.Conv2d(in_channels = 16, out_channels = 32, kernel_size = 5, stride = 1, padding = 2)
    self.relu2 = nn.ReLU()

    # Max pool 2
    self.maxpool2 = nn.MaxPool2d(kernel_size = 2)

    self.fc1 = nn.Linear(32 * 7 * 7, 10)

  def forward(self, x):
    # input: x, size (num_img, 28, 28)

    # Convolution 1
    # O = (28 - 5 + 2*2)/ 1 + 1 = 28
    # output: size (num_img, 16, 28, 28)
    out = self.cnn1(x)
    out = self.relu1(out)

    # Max pool 1
    # O = 28 / 2 = 14
    # output: size (num_img, 16, 14, 14)
    out = self.maxpool1(out)

    # Convolution 2
    # O = (14 - 5 + 2*2)/ 1 + 1 = 14
    # output: size (num_img, 32, 14, 14)
    out = self.cnn2(out)
    out = self.relu2(out)

    # Max pool 2
    # O = 14 / 2 = 7
    # output: size (num_img, 32, 7, 7)
    out = self.maxpool2(out)

    # Resize
    # Original size: (num_img, 32, 7, 7)
    # out.size(0): num_img
    # New out size: (num_img, 32*7*7)
    out = out.view(out.size(0), -1)

    # Linear function (readout)
    # output: size (num_img, 10)
    out = self.fc1(out)

    # log softmax
    out = F.log_softmax(out, dim=1)

    return out

model = victim_CNN()
criterion = nn.CrossEntropyLoss()
learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# now let us train the dataset. We'll test intermittently (this is perfectly customizable!)
iter = 0
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        # Load images
        images = images.requires_grad_()

        # Clear gradients w.r.t. parameters
        optimizer.zero_grad()

        # Forward pass to get output/logits
        outputs = model(images)

        # Calculate Loss: softmax --> cross entropy loss
        loss = criterion(outputs, labels)

        # Getting gradients w.r.t. parameters
        loss.backward()

        # Updating parameters
        optimizer.step()

        iter += 1

        if iter % 500 == 0:
            # Calculate Accuracy
            correct = 0
            total = 0
            # Iterate through test dataset
            for images, labels in test_loader:
                # Load images
                images = images.requires_grad_()

                # Forward pass only to get logits/output
                outputs = model(images)

                # Get predictions from the maximum value
                _, predicted = torch.max(outputs.data, 1)

                # Total number of labels
                total += labels.size(0)

                # Total correct predictions
                correct += (predicted == labels).sum()

            accuracy = 100 * correct / total

            # Print Loss
            print('Iteration: {}. Loss: {}. Accuracy: {}'.format(iter, loss.item(), accuracy))

            if iter == n_iters:
              break




Iteration: 500. Loss: 0.40693214535713196. Accuracy: 86.7300033569336
Iteration: 1000. Loss: 0.19140729308128357. Accuracy: 92.87000274658203


## Inject the bad data into the trained dataset


In [30]:

'''
total_accuracy = []
total_ex = []
epsilons = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 1.0]
for epsilon in epsilons:
  accuracy, ex = perturbator(model, device, test_loader, epsilon)
  total_accuracy.append(accuracy)
  total_ex.append(ex)
'''

'\ntotal_accuracy = []\ntotal_ex = []\nepsilons = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 1.0]\nfor epsilon in epsilons:\n  accuracy, ex = perturbator(model, device, test_loader, epsilon)\n  total_accuracy.append(accuracy)\n  total_ex.append(ex)\n'

# Technique: Black-box adversarial attacks

Say we don't have any information about our model's weights. Instead we shall construct our own neural network, which then learns from the **output** of the original network that we obtain upon feeding it synthetic training data. In theory, an adversarial attack against this intermediary network is also a suitable attack against the original. This notion is called **transferability.**


Main idea is from the following paper:

**Nicolas Papernot, Patrick McDaniel, Ian Goodfellow, Somesh Jha, Z. Berkay Celik, and Ananthram Swami. 2017. Practical Black-Box Attacks against Machine Learning. In Proceedings of the 2017 ACM on Asia Conference on Computer and Communications Security (ASIA CCS '17). Association for Computing Machinery, New York, NY, USA, 506–519. https://doi.org/10.1145/3052973.3053009**

The basic attack philosophy is as follows:

**STEP 1.**
Retrieve our "pretrained" model.

**STEP 2.**
Instantiate our surrogate model.

**STEP 3.**
Randomly select images from the MNIST dataset, and query the original model and assign labels.

**STEP 4.**
Build our NEW dataset from the output, iteratively.

**STEP 5.**
Train our surrogate model on the dataset we've handcrafted.

**STEP 6.**
Mount the FGSM attack from earlier.

In [31]:
# get a pretrained model
from torchvision.models import resnet50, ResNet50_Weights
from tqdm import tqdm
import torch.nn.functional as F
import random

# we'll be using our newly trained model. AVERT YOUR EYES! PRETEND WE DON'T KNOW ANYTHING ABOUT IT!
vmodel = victim_CNN()
final_pred = []

def compute_jacobian(model, X):
    '''
    Get the jacobians of the surrogate with respect to the input samples
    X: torch.Tensor w/ shape (batch, 1, w, h) in [0,1]
    returns: jacobian w/ shape (batch, num_classes, 1, w, h)
    '''
    model.eval()
    X = X.requires_grad(True)
    batch_size = X.shape[0]
    num_classes = model(X).shape[1]

    jacobian = torch.zeros(batch_size, num_classes, *X.shape[1:], device=X.device)
    for i in range(num_classes):
      model.zero_grad()

      outputs = model(X)
      class_outputs = outputs[:,i]
      grads = torch.autograd.grad(
          outputs = class_outputs,
          inputs = X,
          grad_outputs = torch.ones_like(class_outputs),
          retain_graph = True,
          create_graph = False
      )[0]
      jacobian[:,i] = grads
      return jacobian

def query_model(model, imgs):
  '''
  This seems a bit redundant but regardless it's being written here.
  This will just test the model against a batch of images and spit out the estimations
  '''
  images = torch.stack([img for img, _ in imgs])
  labels = torch.tensor([label for _, label in imgs])
  with torch.no_grad():
    outputs = model(images)
    preds = outputs.max(1, keepdims=True)[1]
  return preds.tolist()

def choose_images(img_num):
  '''
  Randomly select img_num images from our dataset
  '''
  indx = random.sample(range(len(test_dataset)),img_num)
  imgs = [test_dataset[i] for i in indx]
  return imgs

class black_box_data_poisoner():
  '''
  Jacobian-based data augmentation method.
  :param model:  surrogate torch model
  :param X:      image data to be augmented
  :param lmb:  hyperparameter step size
  :param
  '''
  def __init__(self, X, lmb, model, victim):
    self.model = model
    self.victim = victim
    self.X = X.copy()
    self.lmb = lmb
    self.num_classes = 10
    self.y = np.empty([0])

  def _jacobian(self, X):
    X = torch.tensor(X, dtype=torch.float32).permute(0,3,1,2)
    J = compute_jacobian(self.model, X)
    return J.cpu().numpy()

  def _augment(self):
    '''
    Uses Jacobian data augmentation technique; this is
    similar to FGSM.
    :param X: selection of images to be augmented
    '''
    J = self._jacobian(self.X)
    J = np.array([J[i,j] for i,j in enumerate(self.y)])
    X_aug = self.X + self.lmb * np.sign(J)
    X_aug = np.clip(X_aug, 0, 1)
    self.X = np.concatenate([self.X, X_aug])
    return

  def _update_labels(self):
    '''
    Build the dataset from the output of the victim NN
    '''
    if np.any(self.y):
      label_me = self.X[-(self.y.shape[0]):]
      y_aug = query_model(self.victim, label_me)
      self.y = np.concatenate([self.y, y_aug])
    else:
      self.y = query_model(self.victim, self.X)
    return

  def _train_model(self):
    '''
    Train the surrogate model
    '''
    num_epochs = 5
    batch_size = 10
    X = self.X
    y = self.y


    return


class surrogate_CNN(nn.Module):
  '''
  Link to paper:
  https://arxiv.org/pdf/1602.02697.pdf
  '''
  def __init__(self):
    super(surrogate_CNN, self).__init__()
    self.cnn1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=2, padding="same")
    self.maxpool1 = nn.MaxPool2d(kernel_size=2)
    self.cnn2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=2,padding=0)
    self.maxpool2 = nn.MaxPool2d(kernel_size=2)
    self.flatten_dim = 64 * 6 * 6
    self.fc1 = nn.Linear(self.flatten_dim, 200)
    self.fc2 = nn.Linear(200, 200)
    self.fc3 = nn.Linear(200, 10)

  def forward(self, x):
    x = self.maxpool1(F.relu(self.cnn1(x)))
    x = self.maxpool2(F.relu(self.cnn2(x)))
    x = x.view(x.size(0), -1)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fx3(x)
    output = F.softmax(x,dim=1)
    return output

surrogate_model = surrogate_CNN()
criterion = nn.CrossEntropyLoss()
learning_rate = 0.01
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# randomly choose N images to query the o.g. model and build a new labeled dataset of size dset_size.
N = 10
dset_size = 50
adv_testdata = choose_images(N)
print(f"Generating {N} adversarial samples.")

# set up the augmenter
aug = black_box_data_poisoner(adv_testdata, 0.1, surrogate_model, vmodel)

Generating 10 adversarial samples.


[[6], [6], [6], [6], [5], [6], [9], [9], [9], [6]]